# Código de concatenação
*by Miguel Ferreira*
Agora, criaremos um algoritmo que concatenará todos os datasets gerados pelas ferramentas do envelope de risco.

## 1. Importação

In [1]:
import pandas as pd

epf  = pd.read_parquet("C:/projects/Libellula/data/processed/epf/epf_features.parquet")
qf   = pd.read_parquet("C:/projects/Libellula/data/processed/qf/qf_features.parquet")
cvar = pd.read_parquet("C:/projects/Libellula/data/processed/cvar/cvar_features.parquet")
pdfd = pd.read_parquet("C:/projects/Libellula/data/processed/pdfd/pdfd_features.parquet")
evt  = pd.read_parquet("C:/projects/Libellula/data/processed/evt/evt_features.parquet")


## 2. Teste de alinhamento

In [2]:
dfs = [epf, qf, cvar, pdfd, evt]

base = dfs[0].index

for i, df in enumerate(dfs):
    assert df.index.equals(base), f"Dataset {i} desalinhado"

Nenhum desalinhamento encontrado.

In [3]:
print(len(epf), len(qf), len(cvar), len(pdfd), len(evt))

205 205 205 205 205


Todos os datasets estão do mesmo tamanho. Agora transformamos todas as colunas de todos os datasets em tipo ```gloat64```

In [4]:
for df in [epf, qf, cvar, pdfd, evt]:
    for col in df.columns:
        if df[col].dtype != "float32":
            df[col] = df[col].astype("float32")

In [5]:
for df in [epf, qf, cvar, pdfd, evt]:
    df.index.name = "timestamp"

In [6]:
for df in [epf, qf, cvar, pdfd, evt]:
    assert df.index.name == "timestamp"
    assert df.index.is_monotonic_increasing
    assert not df.index.has_duplicates
    assert all(df.dtypes == "float32")

## 3. Enfim, a concatenação!!!

In [12]:
risk = pd.concat([epf, qf, cvar, pdfd, evt], axis=1)
risk = risk.dropna()

risk = risk.astype("float32")
risk.index.name = "timestamp"

risk

,epf_upper,epf_lower,epf_width,epf_asymmetry,quantile_lower,q25,q50,q75,quantile_upper,quantile_width,...,pdfd_005,pdfd_01,pdfd_02,pdfd_03,pdf_skew,pdf_kurtosis,evt_shape,evt_scale,evt_var,evt_cvar
timestamp,,,,,,,,,,,,,,,,,,,,,
2023-10-11,1.689235,0.506765,1.182471,1.877802e-16,-0.007350,-0.009410,-0.006859,-0.005027,0.006894,0.014244,...,0.142857,0.019841,0.0,0.0,0.450903,1.363133,0.991691,0.001486,0.149126,17.357655
2023-10-12,1.689235,0.506765,1.182471,1.877802e-16,-0.002481,-0.001492,-0.001558,0.001844,0.008259,0.010740,...,0.146825,0.019841,0.0,0.0,0.451473,1.388208,0.991691,0.001486,0.149126,17.357655
2023-10-13,1.689235,0.506765,1.182471,1.877802e-16,-0.002388,0.001441,0.002554,0.004484,0.008464,0.010852,...,0.146825,0.019841,0.0,0.0,0.452873,1.479422,0.991691,0.001486,0.149126,17.357655
2023-10-16,1.689235,0.506765,1.182471,1.877802e-16,-0.004212,-0.002549,-0.001036,0.001298,0.007754,0.011966,...,0.142857,0.015873,0.0,0.0,0.478121,1.507505,0.991691,0.001486,0.149126,17.357655
2023-10-17,1.689235,0.506765,1.182471,1.877802e-16,-0.006360,-0.003552,-0.003048,-0.000126,0.007754,0.014114,...,0.142857,0.015873,0.0,0.0,0.474413,1.502633,0.991691,0.001486,0.149126,17.357655
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-07-17,1.656859,0.530541,1.126318,-1.971421e-16,-0.007441,-0.003989,-0.002933,-0.001696,0.006761,0.014202,...,0.095238,0.003968,0.0,0.0,0.119067,1.603553,1.393293,0.000457,0.205143,-0.505447
2024-07-18,1.657393,0.521807,1.135585,0.000000e+00,-0.006531,-0.001664,0.000717,0.002348,0.006761,0.013292,...,0.095238,0.003968,0.0,0.0,0.123299,1.574841,1.393293,0.000457,0.205143,-0.505447
2024-07-19,1.636608,0.538792,1.097815,0.000000e+00,-0.006531,-0.003305,-0.002746,-0.001162,0.006862,0.013394,...,0.095238,0.003968,0.0,0.0,0.121387,1.605461,1.393293,0.000457,0.205143,-0.505447


In [14]:
print(risk.shape)
print(risk.dtypes.unique())

(205, 25)
[dtype('float32')]


## 4. Salvamento

In [13]:
risk.to_parquet(
    "C:/projects/Libellula/data/processed/combined/risk_features.parquet"
)